In [21]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import os
import urllib3

# Load environment variables from .env file
load_dotenv()

strPresto = ('presto://{username}:{password}@{ipaddress}:{port}/{dbname}/{schema}'
             .format(username=os.getenv('HIVE_SVC_USER'),
                     password=os.getenv('HIVE_SVC_PASS'),
                     ipaddress=os.getenv('HIVE_SVC_ADDRESS'),
                     port=os.getenv('HIVE_SVC_PORT'),
                     dbname=os.getenv('HIVE_SVC_DBNAME'),
                     schema=os.getenv('HIVE_SVC_SCHEMA')))
 
presto_engine = create_engine(strPresto, connect_args={"protocol": "https", "requests_kwargs": {"verify": False}})

# disable certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [22]:
# ------------------------------------------------------------------
# Start date and end dates for dateselectors in queries
# ------------------------------------------------------------------

#start_date = '2025-12-13' 
#end_date = '2026-02-06' 

today = date.today()
diff_to_friday = (4 - today.weekday()) % 7  # Mon=0 ... Fri=4

# Go back 7 weeks since we are counting the current week
eight_weeks_ago = today - timedelta(weeks=7) 

# Find the Saturday of that week (Mon=0 ... Sun=6, Sat=5)
days_since_saturday = (eight_weeks_ago.weekday() - 5) % 7

start_saturday = eight_weeks_ago - timedelta(days=days_since_saturday)
end_friday = today + timedelta(days=diff_to_friday)

#format date as string for Presto query
start_date = start_saturday.strftime('%Y-%m-%d')
end_date = end_friday.strftime('%Y-%m-%d')


In [23]:
# ------------------------------------------------------------------
# ICP Client list as stored in hive.care.expert_performance_metrics.metric (lowercase)
# ------------------------------------------------------------------

icp_client_list = [
    "pss-verizon",
    "pss-at&t",
    "mob-verizon",
    "mob-at&t",
]

In [24]:
# ------------------------------------------------------------------
# Observe scorecard list
# observe scorecards are the groupings of behavior scores to be queried from observe data
# ------------------------------------------------------------------

observe_scorecards = {
    #"HEROES - Sentiment v1.1",
    #"HEROES LITE v.1",
    #"HEROES - Solve v1.2",
    #"HEROES - Serve v1.2",
    #"HEROES - Sell - Smart Offer v1.1",
    "HEROES Auto Scorecard",
    "HEROES Auto QA Reporting - VZW",
    "Customer Sentiment Scorecard V1",
}

In [25]:
# ------------------------------------------------------------------
# Load SQL template for metric query
# ------------------------------------------------------------------

sql_path = "SQL/observe_data_week.sql"  # <- make sure this path is correct

with open(sql_path, "r") as f:
    OBSERVE_SQL_TEMPLATE = f.read()

print("Loaded SQL template:")
print(OBSERVE_SQL_TEMPLATE[:500], "...")

Loaded SQL template:
WITH 
DateSelector AS (
    SELECT *
    FROM (
        VALUES (
            CAST(:start_date AS date),
            CAST(:end_date   AS date),
            :start_date,
            :end_date
        )
    ) AS t ("StartDate","EndDate","StartDateStr","EndDateStr")
),

org_tenure AS (
    SELECT
        wde.emplid,
        CONCAT(wde.last_name, ', ', wde.pref_first_name, ' (', wde.emplid, ')') AS EmpName,
        CONCAT(wds.last_name, ', ', wds.pref_first_name, ' (', wds.emplid, ')') AS Coach,
     ...


In [26]:
### Compile SQL With Literal Binds (Code)

#This uses your proven pattern: bind params + `literal_binds=True`.

#python
# ------------------------------------------------------------------
# Build literal SQL for Presto using SQLAlchemy binds
# This allows us to use expanding=True for metric_list and still
# send flattened literal SQL to Presto.
# ------------------------------------------------------------------

def compile_presto_sql(
    sql_template: str,
    engine,
    start_date,
    end_date,
    icp_client_list,
    observe_scorecards,
):
    """
    Creates literal SQL for Presto by binding parameters and compiling
    with literal_binds=True.
    """
    
    stmt = text(sql_template).bindparams(
        bindparam("start_date", value=start_date),
        bindparam("end_date", value=end_date),
        bindparam("icp_client_list", value=list(icp_client_list), expanding=True),
        bindparam("observe_scorecards", value=list(observe_scorecards), expanding=True),
    )

    compiled = stmt.compile(
        engine,
        compile_kwargs={"literal_binds": True}
    )

    return str(compiled)

In [27]:
# ------------------------------------------------------------------
# Query Presto for metrics, client groups, and date range, returning the
# aggregated metrics DataFrame.
# ------------------------------------------------------------------

def query_observe_presto_group(
    start_date,
    end_date,
    icp_client_list,
    observe_scorecards,
):
    """
    Execute the Presto query for a list of experts over a date range.

    Returns DataFrame with:
      [expert_id, metric, icp_client, site, num, den, calc]
    """

    sql = compile_presto_sql(
        sql_template=OBSERVE_SQL_TEMPLATE,
        engine=presto_engine,
        start_date=start_date,
        end_date=end_date,
        icp_client_list=icp_client_list,
        observe_scorecards=observe_scorecards,
    )

    # Uncomment to debug generated SQL:
    # print(sql)

    with presto_engine.connect() as conn:
        sql_clean = sql.rstrip()
        if sql_clean.endswith(";"):
            sql_clean = sql_clean[:-1]

        df = pd.read_sql(sql_clean, conn)

    # Normalize types for downstream joins
    #if not df.empty:
   #     df["icp_client"] = df["icp_client"].astype(str)
    #    df["observe_scorecard"] = df["observe_scorecard"].str.lower()

    return df

In [28]:
df = query_observe_presto_group(
        start_date,
        end_date,
        icp_client_list,
        observe_scorecards,
    )

In [29]:
print(OBSERVE_SQL_TEMPLATE)

WITH 
DateSelector AS (
    SELECT *
    FROM (
        VALUES (
            CAST(:start_date AS date),
            CAST(:end_date   AS date),
            :start_date,
            :end_date
        )
    ) AS t ("StartDate","EndDate","StartDateStr","EndDateStr")
),

org_tenure AS (
    SELECT
        wde.emplid,
        CONCAT(wde.last_name, ', ', wde.pref_first_name, ' (', wde.emplid, ')') AS EmpName,
        CONCAT(wds.last_name, ', ', wds.pref_first_name, ' (', wds.emplid, ')') AS Coach,
        m.manager,
        m."sr. manager"   AS sr_manager,
        m.director,
        m.mascot,
        wde.job_title, 
        wde.last_hire_dt,
        date_diff('day', wde.last_hire_dt, current_date) AS tenure
    FROM hive.care.l2_asurion_hrprd_dbo_asu_person_worker AS wde
    LEFT JOIN hive.care.l2_asurion_hrprd_dbo_asu_person_worker AS wds
        ON wde.supervisor_id = wds.emplid
       AND wds.current_flag = TRUE
    LEFT JOIN hive.care.expert_mascots AS m
        ON wde.emplid = m.eid
   

In [30]:
# Save to CSV in the same directory

from pathlib import Path

file = Path("../data/raw/weekly/2026-02-16/behavior_scores.csv")   # replace with your filename

if file.exists():
    file.unlink()
    df.to_csv("../data/raw/weekly/2026-02-16/behavior_scores.csv", index=False)
else:
    df.to_csv("../data/raw/weekly/2026-02-16/behavior_scores.csv", index=False)